In [ ]:
import nest_asyncio
nest_asyncio.apply()

import numpy as np

import pyvista as pv
#pyvista.__version__
from pyvista.trame.jupyter import launch_server
await launch_server().ready
pv.set_jupyter_backend('html') 
#pv.set_jupyter_backend('trame') 

### Check that the environment wakis-env succesfully installe the space mkl multithreading packages

In [ ]:
import os
print(f"Threads available: {os.cpu_count()}")

In [ ]:
pv.Report()

In [ ]:
stl1 = pv.read('clara_stl_files/012_LHC_WVM_6L2-316l.stl')
stl2 = pv.read('clara_stl_files/012_LHC_WVM_6L2-cube.stl')
stl3 = pv.read('clara_stl_files/012_LHC_WVM_6L2-cucamera.stl')
stl4 = pv.read('clara_stl_files/012_LHC_WVM_6L2-martensite.stl')
surf= stl1 + stl2 + stl3 + stl4
pl = pv.Plotter()
pl.add_mesh(surf)

In [ ]:
materials = {'Fingers': [1e4, 1., 1e4]}
stl_names = {'Fingers': 'cube'}
basename = 'Fingers'
stl_solids = {m: f'{basename}-{stl_names[m]}.stl' for m in materials}

surf_finger= pv.read('clara_stl_files/012_LHC_WVM_6L2-cube.stl')

surf_finger= surf_finger.scale(1e-3)   
surf_finger= surf_finger.clean(tolerance=1e-7) 
surf_finger= surf_finger.fill_holes(1000)  
surf_finger= surf_finger.compute_normals(inplace=True, auto_orient_normals=True)   

#has to be here again to restart with only this one STL file, and it's grid
Nx, Ny, Nz = 200, 200, 400  
xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
grid = pv.RectilinearGrid(x, y, z)

#dx = np.diff(x)
#stl_tolerance = np.min(dx) * 1e-3
select = grid.select_interior_points(surf_finger, check_surface=False) 
is_inside = select.point_data['selected_points'].view(np.bool_)

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
inside_voxels = select.threshold(0.5, scalars='selected_points')  
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior') 
pl.add_legend()
pl.show()

#--------------------------------------------------------------------------------------------------------------------------

grid.compute_implicit_distance(surf_finger, inplace=True)
dist = grid["implicit_distance"] 

dx, dy, dz = np.diff(grid.x).mean(), np.diff(grid.y).mean(), np.diff(grid.z).mean()
epsilon = np.mean([dx, dy, dz]) * 0.5
material_weight = np.clip(0.5-(dist / (2*epsilon)),0,1)
grid["material_density"] = material_weight
#self.mesh[grid['implicit_distance'] <= 0] = metal_id

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
inside_voxels = grid.threshold(0.5, scalars='material_density')
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior')
#pl.add_mesh(dist, color='red', show_edges=True, label='Grid implicit distances')
pl.add_legend()
pl.show()



#------------------------------------------------------------------------------------------------------------------------------
xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]

voxelized_grid = surf_finger.voxelize_rectilinear(spacing=spacing)
cpos = pv.CameraPosition(position=(15, 3, 15), focal_point=(0, 0, 0), viewup=(0, 0, 0))
voxelized_grid.plot(scalars='mask', show_edges=False, cpos=cpos)
array_name = 'mask'
is_inside = voxelized_grid.cell_data[array_name].view(np.bool_)

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', style='wireframe', opacity=0.3,label="Origional STL")
inside_voxels = voxelized_grid.extract_cells(voxelized_grid.cell_data[array_name].view(np.bool_))
pl.add_mesh(inside_voxels, color='red', label="Voxelized Interior")
pl.add_legend()
pl.show()

### 1) Select_interior_points from PyVista 0.47, instead of select_enclosed_points form PyVista 0.45/46 (In Wakis!)

https://docs.pyvista.org/api/core/_autosummary/pyvista.datasetfilters.select_interior_points

In [ ]:
materials = {'Body': [1e4, 1., 1e4], 'Fingers': [1e4, 1., 1e4], 
             'Transition tube': [1e4, 1., 1e4], 'Spring': [1e4, 1., 1e4]}
stl_names = {'Body': '316l', 'Fingers': 'cube', 
             'Transition tube': 'cucamera', 'Spring': 'martensite'}
basename = '012_LHC_WVM_6L2'
stl_solids = {m: f'{basename}-{stl_names[m]}.stl' for m in materials}



surf = surf.scale(1e-3)   #just to get the same plot as Elena
surf = surf.clean(tolerance=1e-7)  # 'clean' stitches the 'Body' and 'Fingers' STLs together
surf = surf.fill_holes(1000)  # Filling physical holes in the triangles
surf = surf.compute_normals(inplace=True, auto_orient_normals=True)   

Nx, Ny, Nz = 100, 100, 200  
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
grid = pv.RectilinearGrid(x, y, z)

dx = np.diff(x)
stl_tolerance = np.min(dx) * 1e-3
select = grid.select_interior_points(surf, check_surface=False)  # use check_surface=false because otherwise error warning of the holes pops up
is_inside = select.point_data['selected_points'].view(np.bool_)

Plot of edges that doesnt have a neighbour i.e. 'holes':

In [ ]:
'''
from pyvista.trame.jupyter import launch_server
await launch_server().ready


holes = surf.extract_feature_edges(boundary_edges=True, feature_edges=False)  #extracting edges that doesnt have a neghbour
p = pv.Plotter()
p.add_mesh(surf, opacity=0.5, color='w')
p.add_mesh(holes, color='red', line_width=10) # Red lines = STL device geometry "leaking" water
p.show()
'''

In [ ]:
pl = pv.Plotter()
pl.add_mesh(surf, color='cyan', opacity=0.1, style='wireframe', label='Original STLs')
inside_voxels = select.threshold(0.5, scalars='selected_points')  
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior') 
pl.add_legend()
pl.show()



### 2) Compute implicit distance (PyVista 0.24.0 , to define the surface)

https://docs.pyvista.org/api/core/_autosummary/pyvista.datasetfilters.compute_implicit_distance

In [ ]:


grid.compute_implicit_distance(surf, inplace=True)
'''
#test
#-------------------
print(f"dist range: {grid['implicit_distance'].min()} to {grid['implicit_distance'].max()}")
#--------------------------------
'''
dist = grid["implicit_distance"]  #subpixel 'mask', with transition zone defined by scale epsilon

dx, dy, dz = np.diff(grid.x).mean(), np.diff(grid.y).mean(), np.diff(grid.z).mean()
epsilon = np.mean([dx, dy, dz]) * 0.5
material_weight = np.clip(0.5-(dist / (2*epsilon)),0,1)
grid["material_density"] = material_weight
#self.mesh[grid['implicit_distance'] <= 0] = metal_id

pl = pv.Plotter()
pl.add_mesh(surf, color='cyan', opacity=0.1, style='wireframe', label='Original STLs')
inside_voxels = grid.threshold(0.5, scalars='material_density')
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior')

pl.add_legend()
pl.show()

### 3) PyVista 0.46.0, Voxelize

https://docs.pyvista.org/api/core/_autosummary/pyvista.datasetfilters.voxelize_rectilinear

In [ ]:
#2M grid cell resolution
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
spacing = [
    (xmax - xmin) / 100, 
    (ymax - ymin) / 100, 
    (zmax - zmin) / 200
]

# Voxelize directly creates a RectilinearGrid and marks points inside/outside automatically
voxelized_grid = surf.voxelize_rectilinear(spacing=spacing)
cpos = pv.CameraPosition(position=(15, 3, 15), focal_point=(0, 0, 0), viewup=(0, 0, 0))
voxelized_grid.plot(scalars='mask', show_edges=False, cpos=cpos)

#----------------------------------------------------------------------------------------------
#----------------------------------------------------------------------------------------------
#----------------------------------------------------------------------------------------------
# PyVista 0.47 defaults to 'mask'. We use .get() to avoid crashing if the name changes
if 'mask' in voxelized_grid.cell_data:
    array_name = 'mask'
elif 'voxelized' in voxelized_grid.cell_data:
    array_name = 'voxelized'
else:
    print(f"Available arrays: {voxelized_grid.cell_data.keys()}")
    array_name = list(voxelized_grid.cell_data.keys())[0]
#----------------------------------------------------------------------------------------------
#----------------------------------------------------------------------------------------------
#----------------------------------------------------------------------------------------------
is_inside = voxelized_grid.cell_data[array_name].view(np.bool_)

pl = pv.Plotter()
pl.add_mesh(surf, color='cyan', style='wireframe', opacity=0.3,label="Origional STLs")
inside_voxels = voxelized_grid.extract_cells(voxelized_grid.cell_data[array_name].view(np.bool_))
pl.add_mesh(inside_voxels, color='red', label="Voxelized Interior")
pl.add_legend()
pl.show()


# Fingers

In [ ]:
'''
import nest_asyncio
nest_asyncio.apply()

import numpy as np

import pyvista as pv
#pyvista.__version__
from pyvista.trame.jupyter import launch_server
await launch_server().ready
pv.set_jupyter_backend('trame') 
'''


materials = {'Fingers': [1e4, 1., 1e4]}
stl_names = {'Fingers': 'cube'}
basename = 'Fingers'
stl_solids = {m: f'{basename}-{stl_names[m]}.stl' for m in materials}

surf_finger= pv.read('clara_stl_files/012_LHC_WVM_6L2-cube.stl')

surf_finger= surf_finger.scale(1e-3)   
surf_finger= surf_finger.clean(tolerance=1e-7) 
surf_finger= surf_finger.fill_holes(1000)  
surf_finger= surf_finger.compute_normals(inplace=True, auto_orient_normals=True)   

#has to be here again to restart with only this one STL file, and it's grid
Nx, Ny, Nz = 200, 200, 400  
xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
grid = pv.RectilinearGrid(x, y, z)

#dx = np.diff(x)
#stl_tolerance = np.min(dx) * 1e-3
select = grid.select_interior_points(surf_finger, check_surface=False) 
is_inside = select.point_data['selected_points'].view(np.bool_)

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
inside_voxels = select.threshold(0.5, scalars='selected_points')  
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior') 
pl.add_legend()
pl.show()

#--------------------------------------------------------------------------------------------------------------------------

grid.compute_implicit_distance(surf_finger, inplace=True)
dist = grid["implicit_distance"] 

dx, dy, dz = np.diff(grid.x).mean(), np.diff(grid.y).mean(), np.diff(grid.z).mean()
epsilon = np.mean([dx, dy, dz]) * 0.5
material_weight = np.clip(0.5-(dist / (2*epsilon)),0,1)
grid["material_density"] = material_weight
#self.mesh[grid['implicit_distance'] <= 0] = metal_id

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
inside_voxels = grid.threshold(0.5, scalars='material_density')
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior')
#pl.add_mesh(dist, color='red', show_edges=True, label='Grid implicit distances')
pl.add_legend()
pl.show()



#------------------------------------------------------------------------------------------------------------------------------
xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]

voxelized_grid = surf_finger.voxelize_rectilinear(spacing=spacing)
cpos = pv.CameraPosition(position=(15, 3, 15), focal_point=(0, 0, 0), viewup=(0, 0, 0))
voxelized_grid.plot(scalars='mask', show_edges=False, cpos=cpos)
array_name = 'mask'
is_inside = voxelized_grid.cell_data[array_name].view(np.bool_)

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', style='wireframe', opacity=0.3,label="Origional STL")
inside_voxels = voxelized_grid.extract_cells(voxelized_grid.cell_data[array_name].view(np.bool_))
pl.add_mesh(inside_voxels, color='red', label="Voxelized Interior")
pl.add_legend()
pl.show()

# Benchmarking

In [ ]:
stl_cavity = 'data/002_vacuum_cavity.stl' 
stl_shell = 'data/002_lossymetal_shell.stl'
surf = pv.read(stl_shell) + pv.read(stl_cavity)

#---- pre-loaded + clean -----------------------------------------------------
materials = {'Body': [1e4, 1., 1e4], 'Fingers': [1e4, 1., 1e4], 
             'Transition tube': [1e4, 1., 1e4], 'Spring': [1e4, 1., 1e4]}
stl_names = {'Body': '316l', 'Fingers': 'cube', 
             'Transition tube': 'cucamera', 'Spring': 'martensite'}
basename = '012_LHC_WVM_6L2'
stl_solids = {m: f'{basename}-{stl_names[m]}.stl' for m in materials}

stl1 = pv.read('clara_stl_files/012_LHC_WVM_6L2-316l.stl')
stl2 = pv.read('clara_stl_files/012_LHC_WVM_6L2-cube.stl')
stl3 = pv.read('clara_stl_files/012_LHC_WVM_6L2-cucamera.stl')
stl4 = pv.read('clara_stl_files/012_LHC_WVM_6L2-martensite.stl')
surf= stl1 + stl2 + stl3 + stl4
surf = surf.scale(1e-3)   
surf = surf.clean(tolerance=1e-7)  
surf = surf.fill_holes(1000)  
surf = surf.compute_normals(inplace=True, auto_orient_normals=True)   

surf_finger= stl2
surf_finger= surf_finger.scale(1e-3)   
surf_finger= surf_finger.clean(tolerance=1e-7) 
surf_finger= surf_finger.fill_holes(1000)  
surf_finger= surf_finger.compute_normals(inplace=True, auto_orient_normals=True)   

surf_cavity = pv.read('examples/data/001_vacuum_cavity.stl')
surf_cavity= surf_cavity.scale(1e-3)   
surf_cavity= surf_cavity.clean(tolerance=1e-7) 
surf_cavity= surf_cavity.fill_holes(1000)  
surf_cavity= surf_cavity.compute_normals(inplace=True, auto_orient_normals=True)   
#-----------------------------------------------------------------------------

In [ ]:
#---- pre-loaded + clean -----------------------------------------------------
#materials = {'Body': [1e4, 1., 1e4], 'Fingers': [1e4, 1., 1e4], 'Transition tube': [1e4, 1., 1e4], 'Spring': [1e4, 1., 1e4]}
materials = {'Body': [1e4, 1., 1e4], 'Fingers': [1e4, 1., 1e4], 
             'Transition tube': [1e4, 1., 1e4], 'Spring': [1e4, 1., 1e4],
             'cavity': 'vacuum', 'shell': [30, 1.0, 30]}
#stl_names = {'Body': '316l', 'Fingers': 'cube', 'Transition tube': 'cucamera', 'Spring': 'martensite'}
stl_names = {'Body': '316l', 'Fingers': 'cube', 'Transition tube': 'cucamera', 'Spring': 'martensite','cavity': 'stl_cavity', 'shell': 'stl_shell'}
#basename = '012_LHC_WVM_6L2'
stl_solids = {m: f'{basename}-{stl_names[m]}.stl' for m in materials}


stl_cavity = 'data/002_vacuum_cavity.stl' 
stl_shell = 'data/002_lossymetal_shell.stl'
surf = pv.read(stl_shell) + pv.read(stl_cavity)
'''
stl_solids = {'cavity': stl_cavity, 'shell': stl_shell}
stl_materials = {'cavity': 'vacuum', 'shell': [30, 1.0, 30]}
'''


stl1 = pv.read('clara_stl_files/012_LHC_WVM_6L2-316l.stl')
stl2 = pv.read('clara_stl_files/012_LHC_WVM_6L2-cube.stl')
stl3 = pv.read('clara_stl_files/012_LHC_WVM_6L2-cucamera.stl')
stl4 = pv.read('clara_stl_files/012_LHC_WVM_6L2-martensite.stl')
surf= stl1 + stl2 + stl3 + stl4
surf = surf.scale(1e-3)   
surf = surf.clean(tolerance=1e-7)  
surf = surf.fill_holes(1000)  
surf = surf.compute_normals(inplace=True, auto_orient_normals=True)   

surf_finger= stl2
surf_finger= surf_finger.scale(1e-3)   
surf_finger= surf_finger.clean(tolerance=1e-7) 
surf_finger= surf_finger.fill_holes(1000)  
surf_finger= surf_finger.compute_normals(inplace=True, auto_orient_normals=True)   

surf_cavity = pv.read('examples/data/001_vacuum_cavity.stl')
surf_cavity= surf_cavity.scale(1e-3)   
surf_cavity= surf_cavity.clean(tolerance=1e-7) 
surf_cavity= surf_cavity.fill_holes(1000)  
surf_cavity= surf_cavity.compute_normals(inplace=True, auto_orient_normals=True)   
#-----------------------------------------------------------------------------


methods = ['select_interior', 'implicit_dist', 'voxelize']
surfaces = {'6L2': surf,'Fingers': surf_finger, 'Cavity': surf_cavity}

for m in methods:
    for name, surf in surfaces.items():
        
        xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
        spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]
        x = np.linspace(xmin, xmax, 101)
        y = np.linspace(ymin, ymax, 101)
        z = np.linspace(zmin, zmax, 201)
        grid = pv.RectilinearGrid(x, y, z)

        if m == 'select_interior':
            #select_interior_points (Point-based)
            select = grid.select_interior_points(surf, check_surface=False)
            voxelized_grid = select.point_data_to_cell_data()
            voxelized_grid.cell_data['mask'] = voxelized_grid.cell_data['selected_points']
            
        elif m == 'implicit_dist':
            #implicit_distances (Distance-based), possible to use epsilon
            grid.compute_implicit_distance(surf, inplace=True)
            grid_cells = grid.point_data_to_cell_data()
            grid_cells.cell_data['mask'] = grid_cells.cell_data['implicit_distance'] <= 0
            voxelized_grid = grid_cells
            
        elif m == 'voxelize':
            #voxelize_rectilinear (Direct voxelization)
            voxelized_grid = surf.voxelize_rectilinear(spacing=spacing)   
        
        print(f"\nEvaluating method: {m} | surface: {name}")
        vol_in, vol_out, area = benchmark(voxelized_grid, surf)
        
        mat_val = materials[name][0]
        grid_result = check_subpixel_fraction(grid, surf, mat_val, spacing)
        



